# [✍️ Introduction to Data Modification (DML)](https://learn.microsoft.com/en-us/training/modules/modify-data-with-transact-sql/1-introduction)

Up to this point, we have focused on **Data Query Language (DQL)**—using `SELECT` statements to retrieve, filter, and summarize data for reporting and analysis.

However, databases are dynamic. When developing applications or managing backend systems, you will frequently need to add new records, fix existing ones, or remove old ones. This is known as **Data Manipulation Language (DML)**.

---

## 🎯 Module Objectives

In this module, you will learn the foundational concepts of modifying data within a relational database, specifically how to:

1. **Insert Data:** Add new rows into a table (`INSERT`).
2. **Generate Automatic Values:** Understand how SQL Server handles auto-incrementing identity columns and default values during insertion.
3. **Update Data:** Modify existing records in a table (`UPDATE`).
4. **Delete Data:** Remove unwanted or obsolete rows from a table (`DELETE`).
5. **Merge Data:** Use a single, powerful statement to perform inserts, updates, and deletes simultaneously based on comparing two tables (`MERGE`).

```mermaid
graph TD
    classDef read fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef create fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;
    classDef update fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef delete fill:#ffebee,stroke:#c62828,stroke-width:2px;
    classDef merge fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px;
    classDef db fill:#f5f5f5,stroke:#333,stroke-width:3px;

    subgraph Data Modification Operations
        direction TB
        I["INSERT\n(Create new rows)"]:::create
        U["UPDATE\n(Modify existing rows)"]:::update
        D["DELETE\n(Remove existing rows)"]:::delete
        M["MERGE\n(Synchronize tables:\nInsert, Update, or Delete)"]:::merge
    end

    subgraph Database
        DB[("Relational Database\n(Tables)")]:::db
    end

    I -->|Adds data| DB
    U -->|Changes data| DB
    D -->|Removes data| DB
    M -->|Syncs data| DB
    
    note["Note: SELECT (Read) operations are covered in previous modules."]
```

# Inserting Data into Tables

## The `INSERT` Statement
While `SELECT` is used to retrieve data, the **`INSERT`** statement is used to add one or more new rows of data into an existing table. 

Transact-SQL provides several forms of the `INSERT` statement depending on where your data is coming from:
1. **`INSERT VALUES`**: Inserting literal, manual values.
2. **`INSERT SELECT`**: Inserting data retrieved from another table or query.
3. **`SELECT INTO`**: Creating a brand new table and inserting data into it simultaneously.

```mermaid
graph TD
    A[Ways to Add Data] --> B[1. INSERT VALUES]
    A --> C[2. INSERT ... SELECT]
    A --> D[3. SELECT ... INTO]

    B --> B1[Provide hardcoded values <br> 1 or more rows at a time]
    B1 --> B2[Uses existing table]

    C --> C1[Use a SELECT query to <br> generate rows]
    C1 --> C2[Uses existing table]

    D --> D1[Query an existing table to <br> create and populate a NEW table]
    D1 --> D2[Creates a NEW table]
    
    style B fill:#e1f5fe,stroke:#0288d1
    style C fill:#fff3e0,stroke:#f57c00
    style D fill:#e8f5e9,stroke:#2e7d32
```

***

## 1. `INSERT VALUES`: Inserting Literal Data

The most basic form of the `INSERT` statement allows you to specify the exact values you want to add to a table.

### Basic Syntax
```sql
INSERT [INTO] <Table> [(column_list)]
VALUES ([value1], [value2], ...);
```

### Crucial Rules & Best Practices:
1. **Column List**: The `column_list` is optional but **highly recommended**. 
   - *With column list*: You can insert values in any order, and you can omit columns (if they allow NULLs or have defaults).
   - *Without column list*: You **must** provide a value for every single column in the exact order they were defined in the table.
2. **Using `DEFAULT`**: If you use the keyword `DEFAULT`, SQL Server will use the predefined value for that column (e.g., an auto-generated identity, a default constraint, or `NULL` if allowed).
3. **Using `NULL`**: If a column allows nulls and has no default, you can explicitly insert `NULL`.

### 💡 Pro Tip: Discovering Table Structure
If you don't know the columns in a table, run a `SELECT` query with a condition that is always false (`WHERE 1 = 0`). This returns the column headers without returning any actual data!
```sql
SELECT * FROM Sales.Promotion WHERE 1 = 0;
```

***

**Basic INSERT Examples**
```sql
-- ==========================================
-- DISCOVERING TABLE STRUCTURE
-- ==========================================
-- Returns column names without returning any rows
SELECT * FROM Sales.Promotion WHERE 1 = 0;


-- ==========================================
-- BASIC INSERT (With Column List)
-- ==========================================
INSERT INTO Sales.Promotion (PromotionName, StartDate, ProductModelID, Discount, Notes)
VALUES ('Clearance Sale', '01/01/2021', 23, 0.1, '10% discount');


-- ==========================================
-- BASIC INSERT (Omitting Column List)
-- ⚠️ Only do this if you are providing values for EVERY column in exact table order!
-- ==========================================
INSERT INTO Sales.Promotion
VALUES ('Clearance Sale', '01/01/2021', 23, 0.1, '10% discount');
```

***

**DEFAULT, NULL, and Multi-Row Inserts**
```sql
-- ==========================================
-- USING DEFAULT AND NULL EXPLICITLY
-- ==========================================
-- StartDate uses its default (e.g., current date), Notes is explicitly NULL
INSERT INTO Sales.Promotion
VALUES ('Pull your socks up', DEFAULT, 24, 0.25, NULL);


-- ==========================================
-- OMITTING COLUMNS (Must use Column List)
-- ==========================================
-- StartDate and Notes are omitted. They will get DEFAULT or NULL automatically.
INSERT INTO Sales.Promotion (PromotionName, ProductModelID, Discount)
VALUES ('Caps Locked', 2, 0.2);


-- ==========================================
-- MULTI-ROW INSERT (Table Value Constructor)
-- ==========================================
-- Insert multiple rows at once by separating value sets with commas
INSERT INTO Sales.Promotion
VALUES 
('The gloves are off!', DEFAULT, 3, 0.25, NULL),
('The gloves are off!', DEFAULT, 4, 0.25, NULL);
```

***

## 2. `INSERT ... SELECT`: Copying Data from Queries

Instead of typing out literal values, you can use the results of a `SELECT` query to populate your `INSERT`. This is incredibly useful for copying data from one table to another, or archiving data.

### Syntax
```sql
INSERT [INTO] <target_table> [(column_list)]
SELECT <column_list> FROM <source_table> ...;
```

### Key Considerations:
- **No Parentheses**: Unlike a subquery, the `SELECT` statement used here is **not** enclosed in parentheses.
- **Column Matching**: The number of columns and their data types in the `SELECT` list must match the target table (or the specified `column_list`).
- **Stored Procedures**: You can also use `INSERT EXEC` to insert the results of a stored procedure, but be careful as stored procedures can return multiple result sets.

***

**INSERT ... SELECT Example**
```sql
-- ==========================================
-- INSERT ... SELECT
-- ==========================================
-- Creates a new promotion called 'Get Framed' for every product model 
-- that has the word 'frame' in its name.

INSERT INTO Sales.Promotion (PromotionName, ProductModelID, Discount, Notes)
SELECT DISTINCT 'Get Framed', m.ProductModelID, 0.1, '10% off ' + m.Name
FROM Production.ProductModel AS m
WHERE m.Name LIKE '%frame%';
```

***

## 3. `SELECT ... INTO`: Creating and Populating a New Table

The `SELECT ... INTO` statement is a powerful shortcut. It takes the results of a `SELECT` query and uses them to **create a brand new table**, inserting the data in a single step.

### Syntax
```sql
SELECT <column_list>
INTO <new_table_name>
FROM <source_table>;
```

### Crucial Rules:
1. **Creates a New Table**: Unlike `INSERT ... SELECT`, the target table **must not already exist**. `SELECT INTO` creates it on the fly.
2. **Schema Inheritance**: The new table's columns will inherit the exact same names, data types, and nullability properties from the `SELECT` list.
3. **Fails if Exists**: If a table with the specified name already exists, the query will throw an error and fail.

***

**SELECT ... INTO Example**
```sql
-- ==========================================
-- SELECT ... INTO
-- ==========================================
-- Creates a brand new table called 'Sales.Invoice' 
-- and populates it with data from SalesOrderHeader.

SELECT SalesOrderID, CustomerID, OrderDate, PurchaseOrderNumber, TotalDue
INTO Sales.Invoice
FROM Sales.SalesOrderHeader;

-- Note: If you run this twice, the second run will fail 
-- because the 'Sales.Invoice' table already exists!
```

***

## Summary & Decision Guide

When you need to add data to your database, choose the right tool based on your source data and whether the target table already exists.

### Which Insert Method Should You Use?
```mermaid
flowchart TD
    classDef insert fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef select fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef into fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;
    classDef decision fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px;

    A["Need to add data to the database?"] --> B{"Does the target table ALREADY exist?"}:::decision
    
    B -->|No| C["Use SELECT ... INTO\n(Creates the new table automatically)"]:::into
    
    B -->|Yes| D{"Where is the source data coming from?"}:::decision
    
    D -->|Manual / Literal Values| E["Use INSERT ... VALUES\n(Single row or Table Value Constructor for multiple)"]:::insert
    D -->|From another Table / Query| F["Use INSERT ... SELECT\n(Copies data from a source query)"]:::select
```

### Quick Recap:
- **`INSERT VALUES`**: Best for adding a few manual rows. Use a column list for safety.
- **`INSERT SELECT`**: Best for copying/moving data between *existing* tables.
- **`SELECT INTO`**: Best for quickly creating a new backup, staging, or temporary table from a query.


# Generating Automatic Values

## Why Auto-Generate Values?
In relational databases, you often need to automatically generate sequential, unique values for a specific column (like a Primary Key or an Invoice Number). 

Transact-SQL provides two primary mechanisms to achieve this:
1. **The `IDENTITY` Property**: Tied directly to a specific column within a single table.
2. **The `SEQUENCE` Object**: An independent database object that can be shared across multiple tables or columns.

***

## 1. The `IDENTITY` Property

The `IDENTITY` property is defined at the column level during table creation. It automatically generates sequential numeric values for that specific column.

### Rules for IDENTITY:
- **Data Type**: Must be a numeric data type with a scale of 0 (e.g., `INT`, `BIGINT`, `DECIMAL(5,0)`).
- **Uniqueness**: Only **one** column per table can have the `IDENTITY` property. It is frequently used as the `PRIMARY KEY`.
- **Nullability**: `IDENTITY` columns are automatically `NOT NULL`. You cannot define an `IDENTITY` column as `NULL`.
- **Syntax**: `IDENTITY(seed, increment)`. 
  - *Seed*: The starting value (default is 1).
  - *Increment*: The step value added to the previous row (default is 1).

***

**Creating and Using IDENTITY**
```sql
-- ==========================================
-- CREATING A TABLE WITH AN IDENTITY COLUMN
-- ==========================================
-- PromotionID will start at 1 and increment by 1 automatically.
CREATE TABLE Sales.Promotion
(
    PromotionID int IDENTITY(1,1) PRIMARY KEY,
    PromotionName varchar(20),
    StartDate datetime NOT NULL DEFAULT GETDATE(),
    ProductModelID int NOT NULL,
    Discount decimal(4,2) NOT NULL,
    Notes nvarchar(max) NULL
);

-- ==========================================
-- INSERTING DATA INTO AN IDENTITY TABLE
-- ==========================================
-- Notice we DO NOT provide a value for PromotionID.
-- We also don't need a column list because Identity columns are exempt!
INSERT INTO Sales.Promotion
VALUES ('Clearance Sale', '01/01/2021', 23, 0.10, '10% discount');

-- The database engine automatically assigns PromotionID = 1.
-- The next insert will get 2, then 3, and so on.
```

***

## Retrieving and Overriding IDENTITY Values

### Retrieving the Last Generated Value
After an `INSERT`, you often need to know what ID was just generated (e.g., to insert related child records).
- **`SCOPE_IDENTITY()`**: Returns the last identity value generated in the **current session and current scope**. (Highly Recommended).
- **`IDENT_CURRENT('table_name')`**: Returns the last identity value generated for a **specific table**, regardless of session or scope.

### Overriding the Auto-Generated Value
By default, SQL Server prevents you from manually inserting a value into an `IDENTITY` column. To force a specific value, you must temporarily enable `IDENTITY_INSERT`.

### Reseeding
If you need to reset the counter (e.g., after deleting all rows, or to skip a range of numbers), use the `DBCC CHECKIDENT` command.

***

**Retrieving, Overriding, and Reseeding**
```sql
-- ==========================================
-- RETRIEVING THE LAST IDENTITY VALUE
-- ==========================================
-- Returns the ID generated by the last INSERT in this session
SELECT SCOPE_IDENTITY() AS LastGeneratedID;

-- Returns the current identity value for a specific table
SELECT IDENT_CURRENT('Sales.Promotion') AS CurrentPromotionID;


-- ==========================================
-- OVERRIDING IDENTITY VALUES
-- ==========================================
-- Step 1: Turn ON identity insert for the specific table
SET IDENTITY_INSERT Sales.Promotion ON;

-- Step 2: Insert your explicit value (must include the column list!)
INSERT INTO Sales.Promotion (PromotionID, PromotionName, ProductModelID, Discount)
VALUES (100, 'Manual ID Insert', 37, 0.3);

-- Step 3: Turn it OFF to resume automatic generation
SET IDENTITY_INSERT Sales.Promotion OFF;


-- ==========================================
-- RESEEDING THE IDENTITY COUNTER
-- ==========================================
-- Resets the next identity value to start at 500
DBCC CHECKIDENT ('Sales.Promotion', RESEED, 500);
```

***

## 2. The `SEQUENCE` Object

Unlike `IDENTITY`, which is tied to a single table column, a **`SEQUENCE`** is an independent database object. It generates sequential numbers that can be used by **multiple tables** or even just for display purposes.

### Creating and Using a Sequence
1. Create the sequence using `CREATE SEQUENCE`.
2. Retrieve the next value using the `NEXT VALUE FOR` construct.

> ⚠️ **Important Note**: Every time you call `NEXT VALUE FOR`, the sequence advances. Even if you use it in a `SELECT` statement just to display the numbers, those values are "used up" and will not be reused.

***

**Creating and Using SEQUENCE**
```sql
-- ==========================================
-- CREATING A SEQUENCE
-- ==========================================
-- Creates an independent sequence starting at 1000, incrementing by 1
CREATE SEQUENCE Sales.InvoiceNumber 
AS INT
START WITH 1000 
INCREMENT BY 1;

-- ==========================================
-- USING A SEQUENCE IN AN INSERT
-- ==========================================
INSERT INTO Sales.ResellerInvoice
VALUES (
    NEXT VALUE FOR Sales.InvoiceNumber, -- Gets the next sequence value
    2, 
    GETDATE(), 
    'PO12345', 
    107.99
);

-- ==========================================
-- USING SEQUENCE WITH OVER (ORDER BY)
-- ==========================================
-- Generates sequential numbers based on the sort order of the SELECT
SELECT NEXT VALUE FOR Sales.InvoiceNumber OVER (ORDER BY Name) AS NextID,
       ProductID,
       Name
FROM Production.Product;
```

***

## `IDENTITY` vs. `SEQUENCE`: Which Should You Use?

```mermaid
graph TD
    subgraph The IDENTITY Property
    A[Table 1] -->|Tied directly to the table| B(Generates: 1, 2, 3...)
    end

    subgraph The SEQUENCE Object
    C((Shared Sequence Object)) -->|NEXT VALUE FOR| D[Table A]
    C -->|NEXT VALUE FOR| E[Table B]
    C -->|NEXT VALUE FOR| F[Table C]
    end
    
    style A fill:#e1f5fe,stroke:#0288d1
    style B fill:#d4edda,stroke:#2e7d32
    style C fill:#fff3e0,stroke:#f57c00
    style D fill:#f3e5f5,stroke:#8e24aa
    style E fill:#f3e5f5,stroke:#8e24aa
    style F fill:#f3e5f5,stroke:#8e24aa
```

Both tools generate sequential numbers, but they serve different architectural needs.

| Feature | `IDENTITY` Property | `SEQUENCE` Object |
| :--- | :--- | :--- |
| **Scope** | Tied to a **single column** in a **single table**. | Independent database object; can be shared across **multiple tables**. |
| **Retrieval** | Generated automatically during `INSERT`. | Must explicitly call `NEXT VALUE FOR`. |
| **Updates** | **Protected**. Cannot update an `IDENTITY` column. | Can be updated (if not defined as `NO UPDATE`). |
| **Sorting** | Values are generated in insertion order. | Can use `OVER (ORDER BY ...)` to assign values based on a specific sort order. |
| **Bulk Allocation**| Generates one value at a time. | Can allocate a range of values at once using `sp_sequence_get_range`. |
| **Caching/Changes**| Defined at table creation; hard to change. | Can be altered on the fly (e.g., change increment, restart, cache size). |

### Visualizing the Architecture Difference
```mermaid
graph TD
    classDef table fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef identity fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px;
    classDef sequence fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef db fill:#f5f5f5,stroke:#333,stroke-width:3px;

    subgraph Database Level
        SEQ["SEQUENCE Object\n(Independent)"]:::sequence
    end

    subgraph Table Level
        T1["Table 1: Direct Sales"]:::table
        T2["Table 2: Reseller Sales"]:::table
        T3["Table 3: Product Catalog"]:::table
        
        ID1["IDENTITY Column\n(Tied to Table 1)"]:::identity
    end

    T1 --- ID1
    
    SEQ -->|"Shared across tables"| T1
    SEQ -->|"Shared across tables"| T2
    SEQ -->|"Can be used in SELECT"| T3
```

***

## Decision Guide: Choosing the Right Tool

```mermaid
flowchart TD
    classDef identity fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px;
    classDef sequence fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef decision fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px;

    A["Need to auto-generate sequential numbers?"] --> B{"Does the number need to be shared across MULTIPLE tables?"}:::decision
    
    B -->|Yes| C["Use SEQUENCE"]:::sequence
    
    B -->|No| D{"Do you need to assign numbers based on a specific SORT ORDER?"}:::decision
    D -->|Yes| C
    
    D -->|No| E{"Do you need to reserve MULTIPLE numbers at the exact same time?"}:::decision
    E -->|Yes| C
    
    E -->|No| F["Use IDENTITY\n(Simple, table-specific, protected from updates)"]:::identity
```

### Summary
- Use **`IDENTITY`** for simple, table-specific primary keys where the sequence doesn't need to be shared or manipulated outside of standard inserts.
- Use **`SEQUENCE`** when you need a centralized pool of numbers shared across tables, need to pre-allocate ranges, or need to generate numbers based on a specific `ORDER BY` clause.


# Updating Data in Tables

## The `UPDATE` Statement
The `UPDATE` statement in T-SQL is used to modify existing data in a table. It operates on a set of rows, which can be targeted using a `WHERE` clause or defined via a `JOIN`. 

The `SET` clause specifies which columns are to be modified and provides the new values (which can be expressions, `DEFAULT`, or `NULL`). The `WHERE` clause in an `UPDATE` statement has the exact same structure and filtering capabilities as a `WHERE` clause in a `SELECT` statement.

### Basic Syntax
```sql
UPDATE <TableName>
SET 
    <ColumnName1> = { expression | DEFAULT | NULL },
    <ColumnName2> = { expression | DEFAULT | NULL }
WHERE <search_conditions>;
```

***

**Basic UPDATE Examples**
```sql
-- ==========================================
-- UPDATE A SINGLE COLUMN
-- ==========================================
-- Modifies the notes for a specific promotion based on its ID
UPDATE Sales.Promotion
SET Notes = '25% off socks'
WHERE PromotionID = 2;


-- ==========================================
-- UPDATE MULTIPLE COLUMNS
-- ==========================================
-- Modifies both the Discount and Notes fields simultaneously.
-- Notice the use of the REPLACE() string function to update existing text.
UPDATE Sales.Promotion
SET Discount = 0.2, 
    Notes = REPLACE(Notes, '10%', '20%')
WHERE PromotionName = 'Get Framed';
```

***

## ⚠️ The Danger of the Missing `WHERE` Clause

It is critically important to note that an `UPDATE` statement **without** a corresponding `WHERE` clause will update **all the rows in the table**. 

### Best Practice: Test Before You Update
Before executing an `UPDATE` statement, always run a `SELECT` statement using the exact same `FROM` and `WHERE` clauses. This allows you to verify that you are targeting the correct rows before making permanent changes to the data.

```sql
-- Step 1: Run this first to see what will be affected!
SELECT * FROM Sales.Promotion WHERE PromotionName = 'Get Framed';

-- Step 2: Once verified, run the UPDATE
UPDATE Sales.Promotion SET Discount = 0.2 WHERE PromotionName = 'Get Framed';
```

***

## Updating Data Based on Another Table

The `UPDATE` statement supports a **`FROM`** clause, enabling you to modify data in a target table based on the results of a query or a `JOIN` with another table. This is highly useful when you need to synchronize or update data across related tables.

### Visualizing UPDATE with FROM
```mermaid
flowchart TD
    classDef target fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef source fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef join fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;
    classDef update fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px;

    T["Target Table\nSales.Promotion"]:::target
    S["Source Table\nProduct.ProductModel"]:::source
    
    T --> J{"JOIN Condition\nPromotion.ProductModelID \n= Model.ProductModelID"}:::join
    S --> J
    
    J --> U["UPDATE Target Table\nSET Notes = Source.Name"]:::update
```

***

**UPDATE with FROM Clause**
```sql
-- ==========================================
-- UPDATE WITH FROM CLAUSE (JOIN)
-- ==========================================
-- Updates the Notes column in Sales.Promotion using the Name 
-- retrieved from the Product.ProductModel table.

UPDATE Sales.Promotion
SET Notes = FORMAT(Discount, 'P') + ' off ' + m.Name
FROM Product.ProductModel AS m
WHERE Notes IS NULL
    AND Sales.Promotion.ProductModelID = m.ProductModelID;
```

***

## Summary

- The **`UPDATE`** statement modifies existing rows in a table.
- The **`SET`** clause defines the columns to change and their new values. You can update multiple columns by separating them with commas.
- The **`WHERE`** clause filters which rows are updated. **Omitting it will update the entire table.**
- You can use T-SQL functions (like `REPLACE()`, `FORMAT()`, or math expressions) inside the `SET` clause to calculate new values dynamically.
- The **`FROM`** clause allows you to join the target table to other tables, enabling you to update rows based on data residing in a completely different table.

```mermaid
graph TD
    A[(Table Data)] --> B{WHERE Clause Evaluated}
    
    B -->|Match Found| C[SET Clause Applied]
    B -->|No Match| D[Row Unchanged]
    
    C --> E[Column 1 Updated]
    C --> F[Column 2 Updated]
    
    E --> G([Commit Changes to Table])
    F --> G
    
    style B fill:#fff3e0,stroke:#f57c00
    style C fill:#e1f5fe,stroke:#0288d1
    style G fill:#d4edda,stroke:#2e7d32
```


# Deleting Data from Tables

## The `DELETE` Statement
Just as the `INSERT` statement always adds whole rows to a table, the **`DELETE`** statement always removes entire rows. 

`DELETE` operates on a set of rows, which can be targeted using a condition in a `WHERE` clause or defined via a `JOIN`. The `WHERE` clause in a `DELETE` statement has the exact same structure and filtering capabilities as a `WHERE` clause in a `SELECT` statement.

### Basic Syntax
```sql
DELETE [FROM] <TableName>
WHERE <search_conditions>;
```
*(Note: The `FROM` keyword is optional in T-SQL for a basic delete, but it is highly recommended for readability and ANSI compliance.)*

***

**Basic DELETE Examples**
```sql
-- ==========================================
-- DELETE WITH A WHERE CLAUSE
-- ==========================================
-- Removes all products from the table that have been discontinued.
-- (Assuming 'discontinued = 1' means the product is no longer available)

DELETE FROM Production.Product
WHERE discontinued = 1;


-- ==========================================
-- BEST PRACTICE: Test Before You Delete
-- ==========================================
-- Step 1: Run this SELECT first to see exactly what will be affected!
SELECT * FROM Production.Product WHERE discontinued = 1;

-- Step 2: Once verified, run the DELETE
DELETE FROM Production.Product WHERE discontinued = 1;
```

***

## ⚠️ The Danger of the Missing `WHERE` Clause

It is critically important to keep in mind that a `DELETE` statement **without** a corresponding `WHERE` clause will remove **all the rows** from a table, leaving it completely empty. 

Because of this, `DELETE` is almost always used conditionally with a filter in the `WHERE` clause. Use the `DELETE` statement with extreme caution!

***

## Removing All Rows: `TRUNCATE TABLE`

If your actual goal is to remove **all** the rows and leave an empty table structure behind, you should use the **`TRUNCATE TABLE`** statement instead of a `DELETE` without a `WHERE` clause.

### Key Characteristics of `TRUNCATE TABLE`:
1. **No `WHERE` Clause**: It does not allow a `WHERE` clause. It always removes all rows in one single operation.
2. **High Efficiency**: `TRUNCATE TABLE` is significantly faster and more efficient than `DELETE` when emptying a table. This is because `DELETE` removes rows one by one and logs each deletion, whereas `TRUNCATE` deallocates the data pages directly.
3. **Resets Identity**: If the table has an `IDENTITY` column, `TRUNCATE` will automatically reset the seed value back to its original starting point.

### Syntax
```sql
TRUNCATE TABLE <TableName>;
```

***

**TRUNCATE TABLE Example**
```sql
-- ==========================================
-- TRUNCATE TABLE
-- ==========================================
-- Instantly removes ALL rows from the Sales.Sample table.
-- Much faster than DELETE FROM Sales.Sample;

TRUNCATE TABLE Sales.Sample;
```

***

## `DELETE` vs. `TRUNCATE`: Which Should You Use?

When you need to remove data, use this decision guide to choose the right tool for the job.

```mermaid
flowchart TD
    classDef delete fill:#ffebee,stroke:#c62828,stroke-width:2px;
    classDef truncate fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef decision fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px;

    A["Need to remove data from a table?"] --> B{"Do you need to remove ONLY specific rows?"}:::decision
    
    B -->|Yes| C["Use DELETE with a WHERE clause\n(Logs each row deletion, can be rolled back row-by-row)"]:::delete
    
    B -->|No| D{"Do you want to empty the ENTIRE table?"}:::decision
    D -->|Yes| E["Use TRUNCATE TABLE\n(Fast, deallocates pages, resets IDENTITY seeds)"]:::truncate
```

### Quick Comparison Table
| Feature | `DELETE` | `TRUNCATE TABLE` |
| :--- | :--- | :--- |
| **Purpose** | Remove specific rows or all rows. | Remove **all** rows only. |
| **Filtering** | Uses `WHERE` clause. | No `WHERE` clause allowed. |
| **Performance** | Slower (logs individual row deletions). | Faster (deallocates data pages). |
| **Identity Reset**| Does **not** reset `IDENTITY` seed values. | **Resets** `IDENTITY` seed values. |
| **Trigger Firing** | Fires `DELETE` triggers. | Does **not** fire `DELETE` triggers. |

***

## Summary

- The **`DELETE`** statement removes entire rows from a table.
- Always use a **`WHERE`** clause to target specific rows. Omitting it will delete every row in the table.
- **Best Practice**: Always run a `SELECT` statement with your intended `WHERE` clause first to verify which rows will be deleted.
- If you need to empty a table completely, use **`TRUNCATE TABLE`**. It is much faster and more efficient than an unfiltered `DELETE`.
- `TRUNCATE TABLE` cannot be filtered and will reset any `IDENTITY` columns back to their seed values.

```mermaid
graph TD
    A[(Table Data)] --> B{Need to remove data?}
    
    B -->|Remove specific rows| C[DELETE Statement]
    C --> D{WHERE Clause Included?}
    D -->|Yes| E([Deletes ONLY matching rows])
    D -->|No| F([⚠️ Deletes ALL rows slowly])
    
    B -->|Wipe entire table| G[TRUNCATE TABLE Statement]
    G --> H([⚡ Deletes ALL rows instantly <br> No WHERE clause allowed])
    
    style C fill:#e1f5fe,stroke:#0288d1
    style G fill:#fff3e0,stroke:#f57c00
    style F fill:#f8d7da,stroke:#c62828
    style H fill:#d4edda,stroke:#2e7d32
```

# Merging Data Based on Multiple Tables

## What is the MERGE Statement?
In database operations, there is often a need to synchronize two tables. The **`MERGE`** statement is a powerful DML (Data Manipulation Language) option that allows you to perform **inserts, updates, and deletes** on a target table all in a single, atomic statement, based on differences found in a source table.

### Target vs. Source
- **Target Table**: The table that is being modified (the destination).
- **Source Table**: The table (or query) that is used to determine which rows need to be changed (the origin of the truth).

### The Three Scenarios
`MERGE` evaluates the data and routes it into one of three logical paths:
1. **MATCHED**: The source data has a matching row in the target table $\rightarrow$ **UPDATE** the target.
2. **NOT MATCHED [BY TARGET]**: The source data has *no* match in the target table $\rightarrow$ **INSERT** into the target.
3. **NOT MATCHED BY SOURCE**: The target data has *no* match in the source table $\rightarrow$ **DELETE** from the target.

***

## Visualizing the MERGE Logic

The `MERGE` statement acts as a traffic controller, comparing the Source and Target tables and directing each row to the appropriate action based on the `ON` condition.

```mermaid
flowchart TD
    classDef source fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef target fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef match fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;
    classDef noMatchTarget fill:#bbdefb,stroke:#1565c0,stroke-width:2px;
    classDef noMatchSource fill:#ffcdd2,stroke:#c62828,stroke-width:2px;

    S["Source Table\n(Provides the data)"]:::source
    T["Target Table\n(Receives the changes)"]:::target

    S --> C{"Do the rows match?\n(ON condition)"}
    T --> C

    C -->|Yes: WHEN MATCHED| U["UPDATE Target\n(Update existing rows)"]:::match
    C -->|No: WHEN NOT MATCHED BY TARGET| I["INSERT into Target\n(Add new rows from Source)"]:::noMatchTarget
    C -->|No: WHEN NOT MATCHED BY SOURCE| D["DELETE from Target\n(Remove orphaned rows)"]:::noMatchSource
```

***

## General Syntax of the MERGE Statement

The `MERGE` statement combines the logic of `UPDATE`, `INSERT`, and `DELETE` into a single block. 

> ⚠️ **Crucial Rule**: In T-SQL, the `MERGE` statement **must** be terminated with a semicolon (`;`). If you omit it, SQL Server will throw a syntax error.

```sql
MERGE INTO <TargetTable> AS TargetTbl
USING <SourceTableOrQuery> AS SourceTbl
ON (TargetTbl.KeyColumn = SourceTbl.KeyColumn)

-- Scenario 1: Row exists in both tables
WHEN MATCHED THEN 
   UPDATE SET TargetTbl.ColumnA = SourceTbl.ColumnA

-- Scenario 2: Row exists in Source, but not in Target
WHEN NOT MATCHED BY TARGET THEN
   INSERT (ColumnA, ColumnB)
   VALUES (SourceTbl.ColumnA, SourceTbl.ColumnB)

-- Scenario 3: Row exists in Target, but not in Source
WHEN NOT MATCHED BY SOURCE THEN
   DELETE; -- <--- Mandatory semicolon here!
```

*Note: You do not have to use all three clauses. You can omit the `DELETE` or `INSERT` clauses if your business logic doesn't require them.*

***

**Full MERGE Example**
```sql
-- ==========================================
-- FULL MERGE: UPDATE, INSERT, and DELETE
-- ==========================================
-- This example demonstrates all three actions in a single statement.

MERGE INTO schema_name.table_name AS TargetTbl
USING (SELECT * FROM SourceData) AS SourceTbl
ON (TargetTbl.col1 = SourceTbl.col1)

-- If the key matches, update the target
WHEN MATCHED THEN 
   UPDATE SET TargetTbl.col2 = SourceTbl.col2

-- If the key is in Source but not Target, insert it
WHEN NOT MATCHED BY TARGET THEN
   INSERT (col1, col2)
   VALUES (SourceTbl.col1, SourceTbl.col2)

-- If the key is in Target but not Source, delete it
WHEN NOT MATCHED BY SOURCE THEN
   DELETE;
```

***

## Practical Example: Synchronizing Staged Invoices

Imagine your database includes a table of staged invoice updates (`Sales.InvoiceStaging`). This staging table includes a mix of revisions to existing invoices and brand new invoices. 

Instead of writing separate scripts to figure out which ones are new and which ones are updates, you can use a `MERGE` statement to process the entire staging table in one go.

### What this query does:
1. **MATCHED**: If the `SalesOrderID` exists in both tables, it **updates** the existing invoice with the new data.
2. **NOT MATCHED**: If the `SalesOrderID` is in the staging table but *not* in the main invoice table, it **inserts** the new invoice.
*(Notice we omitted the `WHEN NOT MATCHED BY SOURCE` clause because we don't want to delete existing invoices just because they aren't in today's staging file).*

***

**Invoice Staging MERGE**
```sql
-- ==========================================
-- PRACTICAL MERGE: UPDATE and INSERT ONLY
-- ==========================================
-- Synchronizes the main Invoice table with the daily Staging table.

MERGE INTO Sales.Invoice AS i
USING Sales.InvoiceStaging AS s
ON i.SalesOrderID = s.SalesOrderID

-- Action 1: Update existing invoices with new data
WHEN MATCHED THEN
    UPDATE SET i.CustomerID = s.CustomerID,
               i.OrderDate = GETDATE(),
               i.PurchaseOrderNumber = s.PurchaseOrderNumber,
               i.TotalDue = s.TotalDue

-- Action 2: Insert brand new invoices
WHEN NOT MATCHED BY TARGET THEN
    INSERT (SalesOrderID, CustomerID, OrderDate, PurchaseOrderNumber, TotalDue)
    VALUES (s.SalesOrderID, s.CustomerID, s.OrderDate, s.PurchaseOrderNumber, s.TotalDue);
    -- ^ Notice the mandatory semicolon at the end of the MERGE statement!
```

***

## Summary

- The **`MERGE`** statement allows you to synchronize a **Target** table with a **Source** table in a single, atomic operation.
- It evaluates rows based on an `ON` condition and routes them into three possible actions:
  - **`WHEN MATCHED`**: Updates existing rows.
  - **`WHEN NOT MATCHED [BY TARGET]`**: Inserts new rows.
  - **`WHEN NOT MATCHED BY SOURCE`**: Deletes orphaned rows.
- You only need to include the `WHEN` clauses that apply to your specific business logic (e.g., omitting `DELETE` if you only want to add/update).
- **Syntax Rule**: The `MERGE` statement **must** end with a semicolon (`;`).

```mermaid
graph TD
    A[(Source Table <br> Incoming Data)] --> C{"Compare Target & Source <br> ON (Target.ID = Source.ID)"}
    B[(Target Table <br> Existing Data)] --> C
    
    C -->|ID exists in BOTH| D[WHEN MATCHED]
    C -->|ID is in Source, but NOT Target| E[WHEN NOT MATCHED BY TARGET]
    C -->|ID is in Target, but NOT Source| F[WHEN NOT MATCHED BY SOURCE]
    
    D --> G([UPDATE Target Data])
    E --> H([INSERT New Row into Target])
    F --> I([DELETE Row from Target])
    
    style C fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style G fill:#fff3e0,stroke:#f57c00
    style H fill:#d4edda,stroke:#2e7d32
    style I fill:#f8d7da,stroke:#c62828
```